# Deteksi Tipe Tugas — Regresi vs Klasifikasi

Fokus notebook ini **hanya** menentukan tipe tugas tiap dataset. Tidak ada EDA, tidak ada modeling.

Tiga dataset:
1. Telco Customer Churn
2. Wine Quality
3. Medical Cost (Insurance)


In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
from detect_task import classify_target

pd.set_option("display.width", 120)
print("root:", ROOT)


root: /mnt/c/Users/farmq/Downloads/ml


## 1. Load dataset

In [2]:
telco = pd.read_csv(ROOT / "data/raw/telco/WA_Fn-UseC_-Telco-Customer-Churn.csv")
wine = pd.read_csv(ROOT / "data/raw/wine/WineQT.csv")
insurance = pd.read_csv(ROOT / "data/raw/insurance/insurance.csv")

datasets = {
    "Telco Customer Churn": (telco, "Churn"),
    "Wine Quality": (wine, "quality"),
    "Medical Cost (Insurance)": (insurance, "charges"),
}

for name, (df, target) in datasets.items():
    print(f"{name:28s} shape={df.shape}  target={target!r}")


Telco Customer Churn         shape=(7043, 21)  target='Churn'
Wine Quality                 shape=(1143, 13)  target='quality'
Medical Cost (Insurance)     shape=(1338, 7)  target='charges'


## 2. Deteksi tipe tugas

In [3]:
verdicts = {}
for name, (df, target) in datasets.items():
    verdicts[name] = classify_target(name, df[target])

summary = pd.DataFrame([
    {
        "dataset": name,
        "target": target,
        "dtype": v["target_dtype"],
        "n_unique": v["n_unique"],
        "task": v["task"],
        "subtype": v["subtype"],
        "confidence": v["confidence"],
    }
    for (name, (df, target)), v in zip(datasets.items(), verdicts.values())
])
summary


,dataset,target,dtype,n_unique,task,subtype,confidence
0,Telco Customer Churn,Churn,str,2,classification,binary,high
1,Wine Quality,quality,int64,6,classification,multiclass,high
2,Medical Cost (Insurance),charges,float64,1337,regression,continuous,high


## 3. Alasan per dataset

In [4]:
for name, v in verdicts.items():
    print(f"=== {name} -> {v['task']} / {v['subtype']} ({v['confidence']}) ===")
    for reason in v["reasons"]:
        print("  -", reason)
    if v.get("class_balance"):
        print("  distribusi:", v["class_balance"])
    if v.get("target_stats"):
        print("  statistik :", v["target_stats"])
    if v.get("alternative"):
        print("  alternatif:", v["alternative"])
    print()


=== Telco Customer Churn -> classification / binary (high) ===
  - target bertipe str (kategorikal/teks), bukan besaran kontinu
  distribusi: {'No': 5174, 'Yes': 1869}

=== Wine Quality -> classification / multiclass (high) ===
  - target numerik bulat dengan 6 nilai unik (<= 15)
  - nilai [3, 4, 5, 6, 7, 8] berurutan/kontigu -> label punya urutan, ordinal regression layak dipertimbangkan
  - kelas minoritas hanya 0.5% -> pakai class_weight / resampling
  distribusi: {'3': 6, '4': 33, '5': 483, '6': 462, '7': 143, '8': 16}
  statistik : {'min': 3.0, 'max': 8.0, 'mean': 5.657, 'median': 6.0, 'std': 0.8058, 'skew': 0.2868}
  alternatif: ordinal regression

=== Medical Cost (Insurance) -> regression / continuous (high) ===
  - target numerik kontinu: 1337 nilai unik dari 1338 baris
  - distribusi target miring (skew=1.52)
  statistik : {'min': 1121.8739, 'max': 63770.42801, 'mean': 13270.4223, 'median': 9382.033, 'std': 12110.0112, 'skew': 1.5159}



## 4. Kesimpulan

- Telco → **binary classification** (target kategorikal 2 kelas).
- Wine → **multiclass classification**; alternatif ordinal regression karena label berurutan.
- Insurance → **regression** (target kontinu, 1.337 nilai unik dari 1.338 baris).
- Tidak ada dataset yang salah tipe: dua klasifikasi, satu regresi.
